In [5]:
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import utils
import paths as p
import constants as k

# add group and cohort info to log

In [6]:
sessions_all_raw = pd.read_csv(os.path.join(p.LOGS_DIR, 'sessions_official_raw.csv'), index_col=0)

In [7]:
mouse_to_group = {mouse: group for group, mice in k.GROUP_DICT.items() for mouse in mice}
mouse_to_cohort = {mouse: cohort for cohort, mice in k.COHORT_DICT.items() for mouse in mice}

sessions_all_raw['group'] = sessions_all_raw['mouse'].map(mouse_to_group)
sessions_all_raw['cohort'] = sessions_all_raw['mouse'].map(mouse_to_cohort)

In [8]:
sessions_all_raw

,date,mouse,insertion_number,region,nidaq,simultaneous,probe,hemisphere,depth,probe_treatment,...,potential_problems,current_status,final_data_out,sorting_notes,datetime,paramset_idx,num_units,id,group,cohort
0,2024-07-13,RZ034,1,str,2,n,NaN,r,4000,full monte,...,NaN,4_manual curation,have data,should be all good now,2024-07-13 12:58:26,101,47,RZ034_2024-07-13_str,s,5
1,2024-07-14,RZ034,1,str,2,n,NaN,r,4000,DI water,...,NaN,done,have data,NaN,2024-07-14 12:52:46,101,31,RZ034_2024-07-14_str,s,5
2,2024-07-12,RZ036,0,v1,1,y,NaN,l,2000,none,...,NaN,done,have data,NaN,2024-07-12 12:50:31,101,15,RZ036_2024-07-12_v1,s,5
3,2024-07-12,RZ036,1,str,2,y,NaN,l,4000,DI water,...,NaN,done,have data,NaN,2024-07-12 12:50:31,101,45,RZ036_2024-07-12_str,s,5
4,2024-07-13,RZ036,1,str,2,n,NaN,r,4000,none,...,NaN,done,have data,NaN,2024-07-13 14:29:03,101,30,RZ036_2024-07-13_str,s,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,2025-02-21,RZ065,0,v1,1,y,g,l,2000,full monte,...,NaN,done,have data,"no bug, new phy",2025-02-21 11:15:15,101,27,RZ065_2025-02-21_v1,l,7
92,2025-02-21,RZ065,1,str,2,y,j,l,4000,full monte,...,NaN,1_SI Clustering,NaN,NaN,2025-02-21 11:15:15,101,19,RZ065_2025-02-21_str,l,7
93,2025-02-22,RZ065,1,str,2,y,j,r,4000,full monte,...,NaN,1_SI Clustering,NaN,NaN,2025-02-22 13:03:06,101,162,RZ065_2025-02-22_str,l,7
94,2025-02-13,RZ070,1,str,2,y,e,l,4000,full monte,...,NaN,1_SI Clustering,NaN,NaN,2025-02-13 11:40:15,101,2,RZ070_2025-02-13_str,s,7


# add session performance for filtering

In [ ]:
def get_session_length(trials):
    session_start_time = trials['event_start_time'].iloc[0]
    session_end_time = trials['event_end_time'].iloc[-1]
    session_length = session_end_time - session_start_time
    return session_length

def add_session_performance(sessions_all_raw):
    session_performance_list = []
    for _, session_info in sessions_all_raw.iterrows():
        session_id = session_info['id']
        _, trials, _ = utils.get_session_data(session_id)
        
        num_trials = len(trials)
        num_missed = trials.missed.sum()
        percent_missed = num_missed / num_trials * 100
        num_bg_penalty = num_bg_penalty = (trials['num_bg_repeat'] > 0).sum()
        percent_bg_penalty = num_bg_penalty / num_trials * 100

        session_performance_dict = {
            'id': session_id,
            'length': get_session_length(trials),
            'num_trials': num_trials,
            'num_missed': num_missed,
            'percent_missed': percent_missed,
            'num_bg_penalty': num_bg_penalty,
            'percent_bg_penalty': percent_bg_penalty,
            'wait_length_mean': trials.wait_length.mean()
        }
        session_performance_list.append(session_performance_dict)
    session_performance = pd.DataFrame(session_performance_list)
    session_performance_log = pd.merge(sessions_all_raw, session_performance)
    return session_performance_log

In [ ]:
sessions_all = add_session_performance(sessions_all_raw)
sessions_all.to_csv(os.path.join(p.LOGS_DIR, 'sessions_all.csv'))

In [ ]:
sessions_all

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.histplot(
    data=sessions_all['length'],
    kde=True,          # Adds KDE trend line
    stat="density",    # Normalizes histogram to match KDE scale
    bins="auto",       # Auto-selects bin size (or set manually, e.g., bins=30)
    color="skyblue",   # Histogram color
    edgecolor="white", # Edge color for bars
    line_kws={"color": "red", "lw": 2}  # KDE line style
)

# Labels & title
plt.xlabel('Time (s)')
plt.ylabel('Density')
plt.title('Session Length Distribution')

In [ ]:
metrics = [
    ("percent_missed",     "Percent Missed",     "Distribution of Percent Missed by Group"),
    ("percent_bg_penalty", "Percent BG Repeat",  "Distribution of Percent BG Repeat by Group"),
]

for col, xlabel, title in metrics:
    plt.figure(figsize=(7, 4))
    for label, color in [("All", "gray"), ("l", "blue"), ("s", "orange")]:
        data = sessions_all if label == "All" else sessions_all[sessions_all["group"] == label]
        sns.kdeplot(data=data, x=col, label=label, color=color, linewidth=2)
    plt.xlabel(xlabel)
    plt.ylabel("Density")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
sessions_vetted = sessions_all.loc[
    (sessions_all['num_units'] > 1) &  # Not quiet
    (sessions_all['length'] >= 1500)    # Not short
]

In [ ]:
sessions_vetted = sessions_all.loc[
    (sessions_all['num_units'] > c.MIN_UNITS) &
    (sessions_all['length'] >= c.MIN_SESSION_LENGTH)
]

In [ ]:
sessions_vetted.to_csv(os.path.join(p.LOGS_DIR, 'sessions_vetted.csv'))

In [ ]:
sessions_vetted = pd.read_csv(os.path.join(p.LOGS_DIR, 'sessions_vetted.csv'), index_col=0).sort_values('id')

In [ ]:
group_wait_means = sessions_vetted.groupby('group')['wait_length_mean'].mean()
print(group_wait_means)

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.histplot(
    data=sessions_vetted['num_units'],  # Normalizes histogram to match KDE scale
    bins=15,       # Auto-selects bin size (or set manually, e.g., bins=30)
    color="skyblue",   # Histogram color
    edgecolor="white", # Edge color for bars
    line_kws={"color": "red", "lw": 2}  # KDE line style
)

# Labels & title
plt.xlabel('Number of units per insertion')
plt.ylabel('Count')
plt.title('Session Unit Count')
# plt.savefig(os.path.join(p.LOGS_DIR, "session_unit_count.png"))# no time to finish this

# generate units log

In [ ]:
def generate_units_all(sessions_vetted):
    unit_info_list = []
    for _, session_info in sessions_vetted.iterrows():
        session_id = session_info['id']
        _, _, units = utils.get_session_data(session_id)
        for id, spikes in units.items():
            unit_id = f"{session_id}_unit-{id}"
            spiked_trials = spikes['trial_id'].nunique()
            percent_trials_w_spikes = spiked_trials / session_info['num_trials']
            session_fr = len(spikes) / session_info['length']
            unit_info_dict = {
                'session_id': session_id,
                'id': id,
                'unit_id': unit_id,
                'region': session_info['region'],
                'group': session_info['group'],
                'percent_trials_w_spikes': percent_trials_w_spikes,
                'session_fr': session_fr
            }
            unit_info_list.append(unit_info_dict)
    units_all = pd.DataFrame(unit_info_list)
    return units_all

In [ ]:
units_all = generate_units_all(sessions_vetted)
units_all.to_csv(os.path.join(p.LOGS_DIR, 'units_all.csv'))
print(f"{len(units_all)} units")

In [ ]:
units_all = pd.read_csv(os.path.join(p.LOGS_DIR, 'units_all.csv'), index_col=0)

In [ ]:
num_units_v1 = units_all[units_all['region'] == 'v1'].shape[0]
num_units_str = units_all[units_all['region'] == 'str'].shape[0]
print(f"Number of units in v1: {num_units_v1}")
print(f"Number of units in str: {num_units_str}")

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.histplot(
    data=units_all['percent_trials_w_spikes'],
    kde=True,          # Adds KDE trend line
    stat="density",    # Normalizes histogram to match KDE scale
    bins="auto",       # Auto-selects bin size (or set manually, e.g., bins=30)
    color="skyblue",   # Histogram color
    edgecolor="white", # Edge color for bars
    line_kws={"color": "red", "lw": 2}  # KDE line style
)

# Labels & title
plt.xlabel('Percent Trials with Spikes')
plt.ylabel('Density')
plt.title('Percent Trials with Spikes Distribution')

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.histplot(
    data=units_all['session_fr'],
    kde=True,          # Adds KDE trend line
    stat="density",    # Normalizes histogram to match KDE scale
    bins="auto",       # Auto-selects bin size (or set manually, e.g., bins=30)
    color="skyblue",   # Histogram color
    edgecolor="white", # Edge color for bars
    line_kws={"color": "red", "lw": 2}  # KDE line style
)

# Labels & title
plt.xlabel('Firing Rate (Hz)')
plt.ylabel('Density')
plt.title('Session Firing Rate Distribution')

In [ ]:
units_vetted = units_all.loc[units_all['percent_trials_w_spikes'] >= 0.8]
units_vetted.to_csv(os.path.join(p.LOGS_DIR, 'units_vetted.csv'))

print(f"{len(units_vetted)} units")

In [ ]:
units_vetted = units_all.loc[units_all['percent_trials_w_spikes'] >= c.MIN_PERCENT_TRIALS_WITH_SPIKES]
units_vetted.to_csv(os.path.join(p.LOGS_DIR, 'units_vetted.csv'))

print(f"{len(units_vetted)} units")